In [1]:
import requests
import pandas as pd
import time

lista_artistas = [
    # 🇺🇸 EEUU (10)
    "Taylor Swift", "Drake", "Morgan Wallen", "Kendrick Lamar", "The Weeknd",
    "SZA", "Zach Bryan", "Tyler the Creator", "Billie Eilish", "Ariana Grande",
    # 🇪🇸 España (10)
    "Bad Bunny", "Karol G", "Myke Towers", "Quevedo", "Aitana",
    "Rosalía", "Morad", "Melendi", "Enrique Iglesias", "Juanes",
    # 🇦🇷 Argentina (10)
    "Duki", "María Becerra", "Tini Stoessel", "Nicki Nicole", "Cazzu",
    "Lali", "Paulo Londra", "Shakira", "Bizarrap", "Emilia",
]


In [2]:

# ── 1. OBTENER IDs DE LOS 30 ARTISTAS ──────────────────────────────────────

ids_artistas = {}

for artista in lista_artistas:
    respuesta = requests.get(
        "https://api.deezer.com/search/artist",
        params={"q": artista}
    )
    datos = respuesta.json()
    
    if datos["data"]:
        # Filtrar por nombre exacto y coger el de más fans
        coincidencias = [r for r in datos["data"] if r["name"].lower() == artista.lower()]
        
        if coincidencias:
            mejor = max(coincidencias, key=lambda x: x["nb_fan"])
        else:
            mejor = datos["data"][0]
            print(f"⚠️  {artista}: no hay coincidencia exacta, usando '{mejor['name']}'")
        
        ids_artistas[artista] = mejor["id"]
        print(f"✅ {artista}: {mejor['id']} ({mejor['nb_fan']} fans)")
    else:
        print(f"❌ {artista}: no encontrado")
    
    time.sleep(0.3)

print(f"\nTotal artistas encontrados: {len(ids_artistas)}")

✅ Taylor Swift: 12246 (12525024 fans)
✅ Drake: 246791 (23933064 fans)
✅ Morgan Wallen: 7188840 (282430 fans)
✅ Kendrick Lamar: 525046 (6080362 fans)
✅ The Weeknd: 4050205 (14425298 fans)
✅ SZA: 5531258 (1283553 fans)
✅ Zach Bryan: 71855892 (112716 fans)
⚠️  Tyler the Creator: no hay coincidencia exacta, usando 'Tyler, The Creator'
✅ Tyler the Creator: 1194083 (1323964 fans)
✅ Billie Eilish: 9635624 (9112704 fans)
✅ Ariana Grande: 1562681 (13300713 fans)
✅ Bad Bunny: 10583405 (7737100 fans)
✅ Karol G: 5297021 (3357859 fans)
✅ Myke Towers: 12029862 (1679050 fans)
✅ Quevedo: 6705223 (286545 fans)
✅ Aitana: 11928391 (343374 fans)
✅ Rosalía: 554792 (1022514 fans)
✅ Morad: 111130212 (322051 fans)
✅ Melendi: 2697 (609155 fans)
✅ Enrique Iglesias: 2792 (5940659 fans)
✅ Juanes: 70 (1618019 fans)
✅ Duki: 4902904 (817519 fans)
⚠️  María Becerra: no hay coincidencia exacta, usando 'Maria Becerra'
✅ María Becerra: 14343187 (375261 fans)
✅ Tini Stoessel: 12725191 (8657 fans)
✅ Nicki Nicole: 13299379

In [3]:
import requests
import pandas as pd
import time

lista_artistas = [
    # 🇺🇸 EEUU (10)
    "Taylor Swift", "Drake", "Morgan Wallen", "Kendrick Lamar", "The Weeknd",
    "SZA", "Zach Bryan", "Tyler the Creator", "Billie Eilish", "Ariana Grande",
    # 🇪🇸 España (10)
    "Bad Bunny", "Karol G", "Myke Towers", "Quevedo", "Aitana",
    "Rosalía", "Morad", "Melendi", "Enrique Iglesias", "Juanes",
    # 🇦🇷 Argentina (10)
    "Duki", "María Becerra", "Tini Stoessel", "Nicki Nicole", "Cazzu",
    "Lali", "Paulo Londra", "Shakira", "Bizarrap", "Emilia",
]

# ── 1. OBTENER IDs DE LOS 30 ARTISTAS ──────────────────────────────────────

ids_artistas = {}

for artista in lista_artistas:
    respuesta = requests.get(
        "https://api.deezer.com/search/artist",
        params={"q": artista}
    )
    datos = respuesta.json()
    
    if datos["data"]:
        # Filtrar por nombre exacto y coger el de más fans
        coincidencias = [r for r in datos["data"] if r["name"].lower() == artista.lower()]
        
        if coincidencias:
            mejor = max(coincidencias, key=lambda x: x["nb_fan"])
        else:
            mejor = datos["data"][0]
            print(f"⚠️  {artista}: no hay coincidencia exacta, usando '{mejor['name']}'")
        
        ids_artistas[artista] = mejor["id"]
        print(f"✅ {artista}: {mejor['id']} ({mejor['nb_fan']} fans)")
    else:
        print(f"❌ {artista}: no encontrado")
    
    time.sleep(0.3)

print(f"\nTotal artistas encontrados: {len(ids_artistas)}")

# ── 2. EXTRAER CANCIONES DE CADA ARTISTA ───────────────────────────────────

canciones = []
cache_albums = {}  # evita repetir peticiones de álbumes ya consultados

for artista, id_artista in ids_artistas.items():
    print(f"Descargando canciones de {artista}...")
    
    try:
        respuesta = requests.get(
            f"https://api.deezer.com/artist/{id_artista}/top",
            params={"limit": 50},
            timeout=10
        )
        datos = respuesta.json()
    except Exception as e:
        print(f"  ❌ Error con {artista}: {e}")
        time.sleep(5)
        continue
    
    if "data" in datos:
        for cancion in datos["data"]:
            id_album = cancion["album"]["id"]
            
            # Consultar álbum solo si no lo hemos visto antes
            if id_album not in cache_albums:
                try:
                    resp_album = requests.get(
                        f"https://api.deezer.com/album/{id_album}",
                        timeout=10
                    )
                    cache_albums[id_album] = resp_album.json()
                    time.sleep(0.5)
                except Exception as e:
                    print(f"  ⚠️ Error álbum {id_album}: {e}")
                    cache_albums[id_album] = {}
                    time.sleep(3)
            
            album = cache_albums.get(id_album, {})
            
            # Extraer género
            if album.get("genres") and album["genres"]["data"]:
                genero = album["genres"]["data"][0]["name"]
                id_genero = album["genres"]["data"][0]["id"]
            else:
                genero = None
                id_genero = None
            
            canciones.append({
                "id_artista":       id_artista,
                "nombre_artista":   artista,
                "titulo_cancion":   cancion["title"],
                "titulo_album":     cancion["album"]["title"],
                "tipo":             cancion["type"],
                "anio_lanzamiento": album.get("release_date", None),
                "genero":           genero,
                "id_genero":        id_genero
            })
    
    time.sleep(1)

# ── 3. CREAR DATAFRAME Y LIMPIAR ───────────────────────────────────────────

df = pd.DataFrame(canciones)

# Corregir fechas completas usando el cache
def obtener_fecha_completa(row):
    for id_album, album in cache_albums.items():
        if album.get("title") == row["titulo_album"]:
            return album.get("release_date", None)
    return None

df["anio_lanzamiento"] = df.apply(obtener_fecha_completa, axis=1)

# Rellenar nulos
df["anio_lanzamiento"] = df["anio_lanzamiento"].fillna("Desconocido")
df["genero"]           = df["genero"].fillna("Desconocido")
df["id_genero"]        = df["id_genero"].fillna(0)

print(df)
print(f"\nShape: {df.shape}")
print(f"\nNulos:\n{df.isnull().sum()}")

# ── 4. GUARDAR EN CSV ──────────────────────────────────────────────────────

df.to_csv("canciones_deezer.csv", index=False)
print("✅ Guardado en canciones_deezer.csv")

✅ Taylor Swift: 12246 (12525026 fans)
✅ Drake: 246791 (23933065 fans)
✅ Morgan Wallen: 7188840 (282430 fans)
✅ Kendrick Lamar: 525046 (6080362 fans)
✅ The Weeknd: 4050205 (14425298 fans)
✅ SZA: 5531258 (1283553 fans)
✅ Zach Bryan: 71855892 (112716 fans)
⚠️  Tyler the Creator: no hay coincidencia exacta, usando 'Tyler, The Creator'
✅ Tyler the Creator: 1194083 (1323964 fans)
✅ Billie Eilish: 9635624 (9112705 fans)
✅ Ariana Grande: 1562681 (13300717 fans)
✅ Bad Bunny: 10583405 (7737101 fans)
✅ Karol G: 5297021 (3357859 fans)
✅ Myke Towers: 12029862 (1679050 fans)
✅ Quevedo: 6705223 (286545 fans)
✅ Aitana: 11928391 (343374 fans)
✅ Rosalía: 554792 (1022515 fans)
✅ Morad: 111130212 (322051 fans)
✅ Melendi: 2697 (609155 fans)
✅ Enrique Iglesias: 2792 (5940659 fans)
✅ Juanes: 70 (1618019 fans)
✅ Duki: 4902904 (817519 fans)
⚠️  María Becerra: no hay coincidencia exacta, usando 'Maria Becerra'
✅ María Becerra: 14343187 (375261 fans)
✅ Tini Stoessel: 12725191 (8657 fans)
✅ Nicki Nicole: 13299379

In [4]:
print(df.shape)
print(df.isnull().sum())


(1450, 8)
id_artista          0
nombre_artista      0
titulo_cancion      0
titulo_album        0
tipo                0
anio_lanzamiento    0
genero              0
id_genero           0
dtype: int64


In [5]:
# Tabla bonita en Jupyter
from IPython.display import display
pd.set_option("display.max_columns", None)  # mostrar todas las columnas
pd.set_option("display.max_rows", 20)       # mostrar 20 filas

display(df)

,id_artista,nombre_artista,titulo_cancion,titulo_album,tipo,anio_lanzamiento,genero,id_genero
0,12246,Taylor Swift,The Fate of Ophelia,The Life of a Showgirl,track,2025-10-03,Pop,132.0
1,12246,Taylor Swift,Opalite,The Life of a Showgirl,track,2025-10-03,Pop,132.0
2,12246,Taylor Swift,Blank Space,1989 (Deluxe),track,2014-10-27,Pop,132.0
3,12246,Taylor Swift,Shake It Off,1989 (Deluxe),track,2014-10-27,Pop,132.0
4,12246,Taylor Swift,Out Of The Woods (Taylor's Version),1989 (Taylor's Version),track,2023-10-27,Pop,132.0
...,...,...,...,...,...,...,...,...
1445,61094422,Emilia,Uno los Dos,Uno los Dos,track,2023-02-24,Pop,132.0
1446,61094422,Emilia,Diva (feat. Emilia),Diva (feat. Emilia),track,2022-08-05,Rap/Hip Hop,116.0
1447,61094422,Emilia,Guerrero.mp3,Guerrero.mp3,track,2023-06-15,Pop,132.0
1448,61094422,Emilia,Somos Más,Somos Más,track,2026-03-03,Pop,132.0


In [6]:
# Convertir el diccionario de IDs a DataFrame
df_artistas = pd.DataFrame([
    {"id_artista": id_artista, "nombre_artista": artista}
    for artista, id_artista in ids_artistas.items()
])

display(df_artistas)

,id_artista,nombre_artista
0,12246,Taylor Swift
1,246791,Drake
2,7188840,Morgan Wallen
3,525046,Kendrick Lamar
4,4050205,The Weeknd
...,...,...
25,5181388,Lali
26,12637039,Paulo Londra
27,160,Shakira
28,12487862,Bizarrap


In [7]:
pd.set_option("display.max_rows", None)  # mostrar todas las filas
display(df_artistas)

,id_artista,nombre_artista
0,12246,Taylor Swift
1,246791,Drake
2,7188840,Morgan Wallen
3,525046,Kendrick Lamar
4,4050205,The Weeknd
5,5531258,SZA
6,71855892,Zach Bryan
7,1194083,Tyler the Creator
8,9635624,Billie Eilish
9,1562681,Ariana Grande
